In [31]:
import os
import pandas as pd
import itertools
from sklearn.metrics import brier_score_loss
from sklearn.ensemble import RandomForestClassifier

In [32]:
# Load the processed tournament data
tournament_file_path = os.path.join("..", "ProcessedData", "processed_tournament_data.csv")
tournament_data = pd.read_csv(tournament_file_path)

# Define target
y = tournament_data["Win"]


# Define features
features = ['Divison', 'T1Seed', 'T2Seed', 'SeedDifference', 'T1AverageScore', 
            'T1AverageFGM', 'T1AverageFGA', 'T1AverageFGM3', 'T1AverageFGA3', 'T1AverageFTM', 'T1AverageFTA', 'T1AverageOR', 'T1AverageDR', 
            'T1AverageAst', 'T1AverageTO', 'T1AverageStl', 'T1AverageBlk', 'T1AveragePF', 'T1AverageOpponentScore', 'T1AverageOpponentFGM', 
            'T1AverageOpponentFGA', 'T1AverageOpponentFGM3', 'T1AverageOpponentFGA3', 'T1AverageOpponentFTM', 'T1AverageOpponentFTA', 
            'T1AverageOpponentOR', 'T1AverageOpponentDR', 'T1AverageOpponentAst', 'T1AverageOpponentTO', 'T1AverageOpponentStl', 
            'T1AverageOpponentBlk', 'T1AverageOpponentPF', 'T1AveragePointDifference', 'T2AverageScore', 'T2AverageFGM', 'T2AverageFGA', 
            'T2AverageFGM3', 'T2AverageFGA3', 'T2AverageFTM', 'T2AverageFTA', 'T2AverageOR', 'T2AverageDR', 'T2AverageAst', 'T2AverageTO', 
            'T2AverageStl', 'T2AverageBlk', 'T2AveragePF', 'T2AverageOpponentScore', 'T2AverageOpponentFGM', 'T2AverageOpponentFGA', 
            'T2AverageOpponentFGM3', 'T2AverageOpponentFGA3', 'T2AverageOpponentFTM', 'T2AverageOpponentFTA', 'T2AverageOpponentOR', 
            'T2AverageOpponentDR', 'T2AverageOpponentAst', 'T2AverageOpponentTO', 'T2AverageOpponentStl', 'T2AverageOpponentBlk', 
            'T2AverageOpponentPF', 'T2AveragePointDifference', 'T1Rating', 'T2Rating', 'RatingDifference']

# Create dataframe to hold features
X = tournament_data[features]

In [33]:
# Split data into training data and validation data

# Use data prior to 2024 for training
train_X = X[tournament_data.Season < 2024]
train_y = y[tournament_data.Season < 2024]

# Use 2024 data for validation
val_X = X[tournament_data.Season == 2024]
val_y = y[tournament_data.Season == 2024]

# Create and train the Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=1)
rf_model.fit(train_X, train_y)

# Predict probabilities for the validation set
rf_val_predictions = rf_model.predict_proba(val_X)[:, 1]

# Evaluate predictions using Brier Score
val_brier_score = brier_score_loss(val_y, rf_val_predictions)
print(f"Random Forest Validation Brier Score: {val_brier_score:.4f}")


Random Forest Validation Brier Score: 0.1789


In [34]:
# Load stage 2 sample submission file to get the possible matchups for 2025
sample_submission_file_path = os.path.join("..", "RawData", "SampleSubmissionStage2.csv")
sample_submission = pd.read_csv(sample_submission_file_path)

# Extract season and team IDs from the sample submission file
sample_submission[['Season', 'T1TeamID', 'T2TeamID']] = sample_submission['ID'].str.split('_', expand=True).astype(int)

# Determine division based on team IDs (IDs >= 2000 are women, IDs < 2000 are men)
sample_submission['Division'] = (sample_submission['T1TeamID'] >= 2000).astype(int)

In [35]:
# Load season averages for both teams
season_avg_t1_file_path = os.path.join("..", "ProcessedData", "season_avgs_t1.csv")
season_avg_t2_file_path = os.path.join("..", "ProcessedData", "season_avgs_t2.csv")

season_avgs_t1 = pd.read_csv(season_avg_t1_file_path)
season_avgs_t2 = pd.read_csv(season_avg_t2_file_path)


In [36]:
# Merge season averages with the sample submission

# Key columns for merging

t1_keys = ['Season', 'T1TeamID']
t2_keys = ['Season', 'T2TeamID']

# Remove columns that already exist in sample submisssion to avoid duplication

season_avgs_t1_clean = season_avgs_t1[
    [col for col in season_avgs_t1.columns if col in t1_keys or col not in sample_submission.columns]
]

season_avgs_t2_clean = season_avgs_t2[
    [col for col in season_avgs_t2.columns if col in t2_keys or col not in sample_submission.columns]
]

# Merge season averages into sample submission

sample_submission = sample_submission.merge(season_avgs_t1_clean, on=t1_keys, how='left')
sample_submission = sample_submission.merge(season_avgs_t2_clean, on=t2_keys, how='left')

In [37]:
# Use AverageScore if Elo rating or seed are not available

# Check if average score column exists for both teams 
if 'T1Rating' in sample_submission.columns and 'T2Rating' in sample_submission.columns:
    # Use the team's season average score as a proxy instead of omitting them
    sample_submission['T1Rating'] = sample_submission['T1_AverageScore'] 
    sample_submission['T2Rating'] = sample_submission['T2_AverageScore']
else:
    sample_submission['T1Rating'] = 0
    sample_submission['T2Rating'] = 0

# Calculate difference between rating teams
sample_submission['RatingDifference'] = sample_submission['T1Rating'] - sample_submission['T2Rating']
